# MarketStream — Live Demo
Real-time BTC/USDT price direction prediction pipeline.
- **API:** http://100.48.101.97:8000
- **Stack:** Binance WebSocket → Kafka → DuckDB → LightGBM → FastAPI

In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone
import time
import warnings
warnings.filterwarnings('ignore')

API_URL = "http://100.48.101.97:8000"
plt.style.use('dark_background')

## 1. System Health

In [3]:
health = requests.get(f"{API_URL}/health").json()
latency = requests.get(f"{API_URL}/latency").json()
drift = requests.get(f"{API_URL}/drift").json()

print("=== System Health ===")
print(f"Status:        {health['status']}")
print(f"Model version: {health['model_version']}")
print(f"Timestamp:     {health['timestamp']}")
print()
print("=== Pipeline Latency ===")
print(f"Tick lag:      {latency['tick_lag_seconds']}s")
print(f"Gold lag:      {latency['gold_lag_seconds']}s")
print(f"Total lag:     {latency['total_lag_seconds']}s")
print()
print("=== Drift Detection ===")
print(f"Status:        {drift['status']}")
print(f"Up ratio:      {drift['up_ratio']}")
print(f"Down ratio:    {drift['down_ratio']}")
print(f"Total preds:   {drift['total_predictions']}")

=== System Health ===
Status:        ok
Model version: v1
Timestamp:     2026-06-17T07:06:36.132312+00:00

=== Pipeline Latency ===
Tick lag:      35386.5s
Gold lag:      35436.5s
Total lag:     70823.0s

=== Drift Detection ===
Status:        drift_detected
Up ratio:      0.9286
Down ratio:    0.0714
Total preds:   14


## 2. Live Prediction

In [4]:
pred = requests.get(f"{API_URL}/predict").json()

print("=== Live Prediction ===")
print(f"Symbol:     {pred['symbol']}")
print(f"Prediction: {pred['prediction'].upper()}")
print(f"Confidence: {pred['confidence']:.2%}")
print(f"Threshold:  {pred['threshold']}")
print(f"Timestamp:  {pred['timestamp']}")
print()
print("=== Features Used ===")
for k, v in pred['features_used'].items():
    print(f"  {k:<25} {v:.6f}")

=== Live Prediction ===
Symbol:     BTCUSDT
Prediction: DOWN
Confidence: 98.81%
Threshold:  0.7013
Timestamp:  2026-06-17T07:06:42.131746+00:00

=== Features Used ===
  vwap                      67190.973646
  spread_bps_avg            378.763700
  imbalance_avg             0.343837
  depth_ratio               1.278100
  mid_price_avg             67226.610417
  spread_change             193.976723
  imbalance_change          0.727889
  vwap_vs_mid               -0.000530
  rolling_spread_mean       167.938500
  rolling_imbalance_std     0.320282


## 3. Collect 10 Live Predictions

In [ ]:
predictions = []
print("Collecting 10 predictions (one every 5 seconds)...")
for i in range(10):
    p = requests.get(f"{API_URL}/predict").json()
    predictions.append({
        'timestamp': p['timestamp'],
        'prediction': p['prediction'],
        'confidence': p['confidence'],
        'probability': p['probability'],
        'mid_price': p['features_used']['mid_price_avg'],
        'spread_bps': p['features_used']['spread_bps_avg'],
        'imbalance': p['features_used']['imbalance_avg'],
    })
    print(f"  [{i+1}/10] {p['prediction'].upper():4s} confidence={p['confidence']:.2%} price=${p['features_used']['mid_price_avg']:,.2f}")
    if i < 9:
        time.sleep(5)

df = pd.DataFrame(predictions)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print("\nDone.")

  [1/10] DOWN confidence=98.81% price=$67,226.61
  [2/10] DOWN confidence=98.81% price=$67,226.61
  [3/10] DOWN confidence=98.81% price=$67,226.61
  [4/10] DOWN confidence=98.81% price=$67,226.61
  [5/10] DOWN confidence=98.81% price=$67,226.61
  [6/10] DOWN confidence=98.81% price=$67,226.61
  [7/10] DOWN confidence=98.81% price=$67,226.61
  [8/10] DOWN confidence=98.81% price=$67,226.61


## 4. Visualize Predictions

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('MarketStream — Live BTC/USDT Predictions', fontsize=16, fontweight='bold')

# Price + predictions
ax1 = axes[0]
ax1.plot(df['timestamp'], df['mid_price'], color='#00d4ff', linewidth=2, label='Mid Price')
ups = df[df['prediction'] == 'up']
downs = df[df['prediction'] == 'down']
ax1.scatter(ups['timestamp'], ups['mid_price'], color='#00ff88', s=100, zorder=5, label='UP', marker='^')
ax1.scatter(downs['timestamp'], downs['mid_price'], color='#ff4444', s=100, zorder=5, label='DOWN', marker='v')
ax1.set_ylabel('BTC Price (USD)')
ax1.legend()
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Confidence
ax2 = axes[1]
colors = ['#00ff88' if p == 'up' else '#ff4444' for p in df['prediction']]
ax2.bar(df['timestamp'], df['confidence'], color=colors, width=pd.Timedelta(seconds=3))
ax2.axhline(y=0.7013, color='yellow', linestyle='--', alpha=0.7, label='Threshold (0.7013)')
ax2.set_ylabel('Confidence')
ax2.set_ylim(0, 1)
ax2.legend()

# Imbalance
ax3 = axes[2]
ax3.plot(df['timestamp'], df['imbalance'], color='#ff9900', linewidth=2)
ax3.axhline(y=0, color='white', linestyle='--', alpha=0.3)
ax3.set_ylabel('Order Imbalance')
ax3.set_xlabel('Time (UTC)')

plt.tight_layout()
plt.savefig('../models/plots/demo_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to models/plots/demo_predictions.png")

## 5. Model Info

In [ ]:
info = requests.get(f"{API_URL}/model-info").json()
print("=== Model Metadata ===")
for k, v in info.items():
    if k != 'feature_cols':
        print(f"  {k:<25} {v}")
print()
print("=== Feature Columns ===")
for f in info.get('feature_cols', []):
    print(f"  - {f}")